In [1]:
from pathlib import Path
import pandas as pd
import csv

# Set paths.
s_path_data = Path(r"C:\2026_06_26_Hackathon\Data")
s_file = s_path_data / "Police_Incidents_20260626.csv"
s_out_columns = s_path_data / "Police_Incidents_SNIFF_column_summary.csv"
s_out_sample = s_path_data / "Police_Incidents_SNIFF_sample_rows.csv"
s_out_value_counts = s_path_data / "Police_Incidents_SNIFF_value_counts.txt"

# Sniff delimiter and basic file info.
print("File:", s_file)
print("Size MB:", round(s_file.stat().st_size / 1024 / 1024, 2))

with open(s_file, "r", encoding="utf-8-sig", errors="replace", newline="") as f:
    sample_text = f.read(100000)
    dialect = csv.Sniffer().sniff(sample_text)
    delimiter = dialect.delimiter

print("Delimiter:", repr(delimiter))

# Count rows fast without loading whole CSV.
with open(s_file, "r", encoding="utf-8-sig", errors="replace") as f:
    row_count = sum(1 for _ in f) - 1

print("Approx data rows:", row_count)

# Read first sample only.
df = pd.read_csv(s_file, sep=delimiter, nrows=10000, dtype=str, encoding="utf-8-sig", low_memory=False)
df.columns = df.columns.str.strip()

print("\nRows sampled:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.to_list())

# Build column summary.
summary_rows = []
for col in df.columns:
    s = df[col]
    non_null = s.notna().sum()
    blank = s.fillna("").astype(str).str.strip().eq("").sum()
    unique_count = s.nunique(dropna=True)
    example_values = s.dropna().astype(str).str.strip()
    example_values = example_values[example_values != ""].drop_duplicates().head(5).to_list()
    summary_rows.append({
        "column": col,
        "sample_non_null_count": non_null,
        "sample_blank_count": blank,
        "sample_unique_count": unique_count,
        "sample_blank_pct": round(blank / len(df) * 100, 2),
        "example_values": " | ".join(example_values)
    })

df_summary = pd.DataFrame(summary_rows).sort_values(["sample_blank_pct", "sample_unique_count"], ascending=[True, False])
df_summary.to_csv(s_out_columns, index=False)

# Save sample rows.
df.head(500).to_csv(s_out_sample, index=False)

# Write value counts for likely useful low-cardinality columns.
with open(s_out_value_counts, "w", encoding="utf-8") as f:
    f.write(f"File: {s_file}\nRows sampled: {len(df)}\nApprox total rows: {row_count}\n\n")
    for col in df.columns:
        unique_count = df[col].nunique(dropna=True)
        if unique_count <= 50:
            f.write("\n" + "=" * 80 + "\n")
            f.write(f"{col}\n")
            f.write("=" * 80 + "\n")
            f.write(df[col].fillna("[NULL]").astype(str).str.strip().replace("", "[BLANK]").value_counts().head(30).to_string())
            f.write("\n")

# Guess useful crime/location/date columns by name.
keywords = ["date", "year", "time", "offense", "crime", "incident", "ucr", "nibrs", "weapon", "victim", "location", "address", "zip", "beat", "division", "sector", "latitude", "longitude", "x", "y"]
possible_keep_cols = [col for col in df.columns if any(k in col.lower() for k in keywords)]

print("\nLikely useful columns:")
print(possible_keep_cols)

print("\nSaved:")
print(s_out_columns)
print(s_out_sample)
print(s_out_value_counts)

File: C:\2026_06_26_Hackathon\Data\Police_Incidents_20260626.csv
Size MB: 1523.01
Delimiter: ','
Approx data rows: 1494935

Rows sampled: 10000
Columns: 86

Column names:
['Incident Number w/year', 'Year of Incident', 'Service Number ID', 'Watch', 'Call (911) Problem', 'Type of Incident', 'Type  Location', 'Type of Property', 'Incident Address', 'Apartment Number', 'Reporting Area', 'Beat', 'Division', 'Sector', 'Council District', 'Target Area Action Grids', 'Community', 'Date1 of Occurrence', 'Year1 of Occurrence', 'Month1 of Occurence', 'Day1 of the Week', 'Time1 of Occurrence', 'Day1 of the Year', 'Date2 of Occurrence', 'Year2 of Occurrence', 'Month2 of Occurence', 'Day2 of the Week', 'Time2 of Occurrence', 'Day2 of the Year', 'Date of Report', 'Date incident created', 'Offense Entered Year', 'Offense Entered Month', 'Offense Entered Day of the Week', 'Offense Entered Time', 'Offense Entered  Date/Time', 'CFS Number', 'Call Received Date Time', 'Call Date Time', 'Call Cleared Date 